# Teste avaliativo para vaga de bolsista em engenharia/análise de dados

## 1. Extração dos dados

In [1]:
import json
import numpy as np
import os
import pandas as pd
import pickle
import re
import requests
import sqlite3
import time

In [ ]:
# CONFIGURAR PASTA PARA SALVAR OS DADOS DA REQUISIÇÃO
os.makedirs('data', exist_ok=True)


URL = 'https://api.obrasgov.gestao.gov.br/obrasgov/api'
ENDPOINT = 'projeto-investimento'
MAX_ATTEMPT = 2
PAGE_SIZE = 10
UF = 'DF'
page = 0 


# PROJETOS DO DF
projetos_DF = []

# BUSCAR DADOS UTILIZANDO A API FORNECIDA
while True:
    response = requests.get(f'{URL}/{ENDPOINT}?uf={UF}&pagina={page}&tamanhoDaPagina={PAGE_SIZE}')

    # se vir um erro 404 indica que aquela página não existe ou a requisição está falhando
    if response.status_code == 404:
        print(f'Página {page} não existe. Finalizando.')
        break

    # se vir um erro 429, siginifica que meu ip tá sendo negado, preciso aguardar o rate-limit imposto pela api
    elif response.status_code == 429:
        retry_after = int(response.headers.get('x-rate-limit-retry-after-seconds', 2))
        print(f'Muitas requisições! Aguardando {retry_after} segundos...')
        time.sleep(retry_after)
        continue

    try:
        data = response.json()
    except ValueError as e:
        print(f'Erro ao decodificar JSON na página {page}: {e}')
        break

    # se pagina estiver vazia:
    if not data.get("content") or data.get("empty", False):
        print(f'Acabaram os dados para o UF = {UF}. ({page=})')
        break

    for item in data.get('content', []):
        projetos_DF.append(item)

    page += 1
    time.sleep(2)
        

print(f'Dados carregados. Total de dados encontrados = {len(projetos_DF)}.')


# SALVAR DADOS EM JSON:
json_path = os.path.join('data', 'projetos_DF.json')
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(projetos_DF, file, ensure_ascii=False, indent=2)
print(f'Arquivo JSON salvo: {json_path}')


# SALVAR DADOS EM CSV:
csv_path = os.path.join('data', 'projetos_DF.csv')
df = pd.DataFrame(projetos_DF)
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f'Arquivo CSV salvo: {csv_path}')


# SALVAR DADOS EM PICKLE:
pkl_path = os.path.join('data', 'projetos_DF.pkl')
with open(pkl_path, "wb") as file:
    pickle.dump(projetos_DF, file)
print(f'Arquivo Pickle salvo: {pkl_path}')

Acabaram os dados para o UF = DF. (page=84)
Dados carregados. Total de dados encontrados = 834.
Arquivo JSON salvo: data/projetos_DF.json
Arquivo CSV salvo: data/projetos_DF.csv
Arquivo Pickle salvo: data/projetos_DF.pkl


## 2. Busca exploratória dos dados

### 2.1 Visão geral dos dados

In [2]:
# CARREGAR OS DADOS VINDOS DE ALGUM ARQUIVO SALVO ACIMA, PARA ESSE EXEMPLO, IREI USAR OS DADOS VINDOS DO CSV:
df = pd.read_csv('data/projetos_DF.csv')

# quantidade de linhas e colunas:
rows, cols = df.shape

print(f'Dimensões (Linhas x Colunas): {rows} x {cols}')
print(f'Quantidade de linhas: {rows}')
print(f'Quantidade de colunas: {cols}')
print(f'Total de dados: (linhas x colunas) = {rows * cols}\n')

Dimensões (Linhas x Colunas): 834 x 31
Quantidade de linhas: 834
Quantidade de colunas: 31
Total de dados: (linhas x colunas) = 25854



In [3]:
# quantidade de valores nulos
qtd_null = df.isnull().sum()

# porcentagem de valores nulos:
percent_null = qtd_null / len(df) * 100

print("DADOS NULOS POR COLUNAS DOS DADOS OBTIDOS:\n")

for col in df.columns:
    print(f'{col}: {qtd_null[col]} --- ({percent_null[col]:.2f}%)')

DADOS NULOS POR COLUNAS DOS DADOS OBTIDOS:

idUnico: 0 --- (0.00%)
nome: 0 --- (0.00%)
cep: 453 --- (54.32%)
endereco: 445 --- (53.36%)
descricao: 0 --- (0.00%)
funcaoSocial: 0 --- (0.00%)
metaGlobal: 0 --- (0.00%)
dataInicialPrevista: 2 --- (0.24%)
dataFinalPrevista: 2 --- (0.24%)
dataInicialEfetiva: 812 --- (97.36%)
dataFinalEfetiva: 828 --- (99.28%)
dataCadastro: 0 --- (0.00%)
especie: 7 --- (0.84%)
natureza: 0 --- (0.00%)
naturezaOutras: 763 --- (91.49%)
situacao: 0 --- (0.00%)
descPlanoNacionalPoliticaVinculado: 576 --- (69.06%)
uf: 0 --- (0.00%)
qdtEmpregosGerados: 673 --- (80.70%)
descPopulacaoBeneficiada: 674 --- (80.82%)
populacaoBeneficiada: 678 --- (81.29%)
observacoesPertinentes: 718 --- (86.09%)
isModeladaPorBim: 259 --- (31.06%)
dataSituacao: 0 --- (0.00%)
tomadores: 0 --- (0.00%)
executores: 0 --- (0.00%)
repassadores: 0 --- (0.00%)
eixos: 0 --- (0.00%)
tipos: 0 --- (0.00%)
subTipos: 0 --- (0.00%)
fontesDeRecurso: 0 --- (0.00%)


In [4]:
# informações do dataframe:

print('INFORMAÇÕES DO DATAFRAME:\n')
df.info()

INFORMAÇÕES DO DATAFRAME:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 834 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   idUnico                             834 non-null    object
 1   nome                                834 non-null    object
 2   cep                                 381 non-null    object
 3   endereco                            389 non-null    object
 4   descricao                           834 non-null    object
 5   funcaoSocial                        834 non-null    object
 6   metaGlobal                          834 non-null    object
 7   dataInicialPrevista                 832 non-null    object
 8   dataFinalPrevista                   832 non-null    object
 9   dataInicialEfetiva                  22 non-null     object
 10  dataFinalEfetiva                    6 non-null      object
 11  dataCadastro                   

### 2.2 Informações descritivas dos dados

In [5]:
# informações descritivas do dataframe:

print('INFORMAÇÕES DESCRITIVAS DO DATAFRAME:\n')
print('count = quantidade de valores não nulos')
print('unique = quantidade de valores únicos')
print('top = valor mais frequente')
print('freq = frequência do valor mais comum')
print('dtype = tipo do dado (será tratado mais para a frente)')

# visão detalhada por coluna:
for col in df.columns:
    print(f"\n=== Estatísticas da coluna '{col}' ===\n")
    print(df[col].describe(), "\n")

# visão em formato de tabela:
df.describe() 

INFORMAÇÕES DESCRITIVAS DO DATAFRAME:

count = quantidade de valores não nulos
unique = quantidade de valores únicos
top = valor mais frequente
freq = frequência do valor mais comum
dtype = tipo do dado (será tratado mais para a frente)

=== Estatísticas da coluna 'idUnico' ===

count             834
unique            606
top       60895.53-14
freq                4
Name: idUnico, dtype: object 


=== Estatísticas da coluna 'nome' ===

count                                       834
unique                                      570
top       CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE
freq                                          9
Name: nome, dtype: object 


=== Estatísticas da coluna 'cep' ===

count     381
unique     85
top         1
freq       92
Name: cep, dtype: object 


=== Estatísticas da coluna 'endereco' ===

count         389
unique        220
top        BR-010
freq           12
Name: endereco, dtype: object 


=== Estatísticas da coluna 'descricao' ===

count                     

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
count,834,834,381,389,834,834,834,832,832,22,...,116,575,834,834,834,834,834,834,834,834
unique,606,570,85,220,558,447,430,392,424,11,...,3,2,348,48,73,51,15,48,79,461
top,60895.53-14,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,1,BR-010,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,Segurança Pública,Escola de Educação Infantil Tipo B,2025-06-01,2027-06-01,2023-12-15,...,Informações Obras Fundo Nacional de Desenvolvi...,False,2025-07-25,[],[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,[],"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 5, 'descricao': 'Administrativo', 'idE...","[{'id': 36, 'descricao': 'Dragagem, Derrocamen...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
freq,4,9,92,12,9,35,51,39,40,6,...,114,536,81,462,115,373,335,140,114,133


### 2.3 Qualidade dos dados

In [6]:
# qualidade dos dados:

print('QUALIDADE DOS DADOS DO DATAFRAME:\n')

# quantidade de linhas duplicadas:
rows_duplicated = df.duplicated().sum()
print(f'Quantidade de linhas duplicadas: {rows_duplicated}\n')

# total de valores nulos (sendo que o total de valores foi dito acima)
total_null = df.isnull().sum().sum()
print(f'Total de valores nulos: {total_null}\n')

# exemplo de linhas com valores nulos:
rows_nulls = df[df.isnull().any(axis=1)]
print(f'Exemplo de linhas com valores nulos:')
rows_nulls.head()

QUALIDADE DOS DADOS DO DATAFRAME:

Quantidade de linhas duplicadas: 188

Total de valores nulos: 6890

Exemplo de linhas com valores nulos:


,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,NaN,NaN,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,NaN,...,NaN,False,2024-12-20,[],[{'nome': 'DEPARTAMENTO NACIONAL DE INFRAESTRU...,[],"[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 25, 'descricao': 'Rodovia', 'idEixo': 3}]","[{'id': 4, 'descricao': 'Acessos Terrestres', ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
1,42724.53-27,Escola Classe Crixá São Sebastião,NaN,NaN,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,NaN,...,NaN,False,2025-09-05,[],[{'nome': 'SECRETARIA DE ESTADO DE EDUCACAO DO...,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,"[{'id': 4, 'descricao': 'Social'}]","[{'id': 46, 'descricao': 'Educação', 'idEixo':...","[{'id': 84, 'descricao': 'Educação', 'idTipo':...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70.602-600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,NaN,...,NaN,False,2023-02-06,[],[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 1, 'descricao': 'Segurança Pública', '...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,NaN,NaN,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,NaN,...,NaN,False,2023-08-28,[],[{'nome': 'DEPARTAMENTO DE ESTRADAS DE RODAGEM...,"[{'nome': 'MINISTÉRIO DAS CIDADES', 'codigo': ...","[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 24, 'descricao': 'Infraestrutura Urban...","[{'id': 57, 'descricao': 'Obra de Arte Especia...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",NaN,NaN,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,NaN,...,NaN,False,2023-08-29,[],"[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'id': 3, 'descricao': 'Econômico'}, {'id': 3...","[{'id': 31, 'descricao': 'Energia', 'idEixo': ...","[{'id': 95, 'descricao': 'Subestação', 'idTipo...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."


## 3. Tratamento de dados

In [7]:
# TRATAMENTO DOS DADOS

new_df = df.copy()

# datas
DATE_COLS = [
    'dataInicialPrevista', 'dataFinalPrevista',
    'dataInicialEfetiva', 'dataFinalEfetiva',
    'dataCadastro', 'dataSituacao'
]

for date_col in DATE_COLS:
    if date_col in new_df.columns:
        new_df[date_col] = pd.to_datetime(new_df[date_col], errors='coerce')


# valores numericos
NUM_COLS = ['qdtEmpregosGerados', 'populacaoBeneficiada']

for num_col in NUM_COLS:
    if num_col in new_df.columns:
        new_df[num_col] = (
            new_df[num_col]
            .astype(str)
            .str.replace(r'[^0-9,.-]', '', regex=True)
            .str.replace(',', '.', regex=False)
        )
        new_df[num_col] = pd.to_numeric(new_df[num_col], errors='coerce')
        new_df[num_col] = new_df[num_col].fillna(0)

# textos
TEXT_COLS = [
    'cep', 'descricao', 'endereco', 'nome', 'funcaoSocial', 
    'metaGlobal', 'descPopulacaoBeneficiada',
    'observacoesPertinentes', 'descPlanoNacionalPoliticaVinculado'
]
for col in TEXT_COLS:
    if col in new_df.columns:
        new_df[col] = new_df[col].fillna('Não informado')
        
# tratando cep:


def tratar_cep(cep):
    if not cep or cep in ["nan", "NaN", "None"]:
        return ""
    digits = re.sub(r"\D", "", str(cep))
    return digits if len(digits) >= 8 else ""

new_df["cep"] = new_df["cep"].apply(tratar_cep)

        
# booleanos
BOOL_COLS = ['isModeladaPorBim']
for col in BOOL_COLS:
    if col in new_df.columns:
        new_df[col] = new_df[col].fillna(False)
        

# categorias
CATEGORY_COLS = ['uf', 'situacao', 'especie', 'natureza']
for col in CATEGORY_COLS:
    if col in new_df.columns:
        new_df[col] = new_df[col].astype('category')
        

# listas
LIST_COLS = ['tomadores', 'executores', 'repassadores', 'eixos', 'tipos', 'subTipos', 'fontesDeRecurso']

for col in LIST_COLS:
    if col in new_df.columns:
        new_df[col] = new_df[col].apply(lambda x: 'Não informado ou não existe' if x == '[]' else x)

        

new_df.info()
new_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 834 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   idUnico                             834 non-null    object        
 1   nome                                834 non-null    object        
 2   cep                                 834 non-null    object        
 3   endereco                            834 non-null    object        
 4   descricao                           834 non-null    object        
 5   funcaoSocial                        834 non-null    object        
 6   metaGlobal                          834 non-null    object        
 7   dataInicialPrevista                 832 non-null    datetime64[ns]
 8   dataFinalPrevista                   832 non-null    datetime64[ns]
 9   dataInicialEfetiva                  22 non-null     datetime64[ns]
 10  dataFinalEfetiva          

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,,Não informado,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,NaT,...,Não informado,False,2024-12-20,Não informado ou não existe,[{'nome': 'DEPARTAMENTO NACIONAL DE INFRAESTRU...,Não informado ou não existe,"[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 25, 'descricao': 'Rodovia', 'idEixo': 3}]","[{'id': 4, 'descricao': 'Acessos Terrestres', ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
1,42724.53-27,Escola Classe Crixá São Sebastião,,Não informado,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,NaT,...,Não informado,False,2025-09-05,Não informado ou não existe,[{'nome': 'SECRETARIA DE ESTADO DE EDUCACAO DO...,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,"[{'id': 4, 'descricao': 'Social'}]","[{'id': 46, 'descricao': 'Educação', 'idEixo':...","[{'id': 84, 'descricao': 'Educação', 'idTipo':...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70602600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,NaT,...,Não informado,False,2023-02-06,Não informado ou não existe,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 1, 'descricao': 'Segurança Pública', '...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,,Não informado,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,NaT,...,Não informado,False,2023-08-28,Não informado ou não existe,[{'nome': 'DEPARTAMENTO DE ESTRADAS DE RODAGEM...,"[{'nome': 'MINISTÉRIO DAS CIDADES', 'codigo': ...","[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 24, 'descricao': 'Infraestrutura Urban...","[{'id': 57, 'descricao': 'Obra de Arte Especia...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",,Não informado,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,NaT,...,Não informado,False,2023-08-29,Não informado ou não existe,"[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'id': 3, 'descricao': 'Econômico'}, {'id': 3...","[{'id': 31, 'descricao': 'Energia', 'idEixo': ...","[{'id': 95, 'descricao': 'Subestação', 'idTipo...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."


In [8]:
# NORMALIZAR OS DADOS

# renomear colunas:

new_df = new_df.rename(
    columns=lambda x: x.replace('desc', 'descricao') if x.startswith('desc') else x
)

new_df = new_df.rename(
    columns=lambda x: x.replace('qdt', 'quantidade') if x.startswith('qdt') else x
)

new_df = new_df.rename(
    columns=lambda x: x.replace('is', 'eh') if x.startswith('is') else x
)

new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 834 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   idUnico                                  834 non-null    object        
 1   nome                                     834 non-null    object        
 2   cep                                      834 non-null    object        
 3   endereco                                 834 non-null    object        
 4   descricaoricao                           834 non-null    object        
 5   funcaoSocial                             834 non-null    object        
 6   metaGlobal                               834 non-null    object        
 7   dataInicialPrevista                      832 non-null    datetime64[ns]
 8   dataFinalPrevista                        832 non-null    datetime64[ns]
 9   dataInicialEfetiva                       22

In [9]:
# remover duplicatas pelo id, nomr e cep
new_df = new_df.drop_duplicates(subset=['idUnico', 'nome', 'cep'])

new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 606 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   idUnico                                  606 non-null    object        
 1   nome                                     606 non-null    object        
 2   cep                                      606 non-null    object        
 3   endereco                                 606 non-null    object        
 4   descricaoricao                           606 non-null    object        
 5   funcaoSocial                             606 non-null    object        
 6   metaGlobal                               606 non-null    object        
 7   dataInicialPrevista                      604 non-null    datetime64[ns]
 8   dataFinalPrevista                        604 non-null    datetime64[ns]
 9   dataInicialEfetiva                       13 non-

In [10]:
# visualizando dados descritivos numericos e de datas:
new_df.describe()

,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,dataFinalEfetiva,dataCadastro,quantidadeEmpregosGerados,populacaoBeneficiada,dataSituacao
count,604,604,13,4,606,606.000000,6.060000e+02,606
mean,2021-07-30 18:04:46.092715008,2023-06-30 02:15:53.642384128,2022-06-16 16:36:55.384615424,2023-02-15 00:00:00,2023-08-28 17:06:32.079207936,3.867987,1.207205e+04,2023-08-27 20:49:54.059405824
min,2003-06-26 00:00:00,2007-11-09 00:00:00,2018-05-17 00:00:00,2022-03-22 00:00:00,2021-01-20 00:00:00,0.000000,0.000000e+00,2007-11-09 00:00:00
25%,2020-11-21 06:00:00,2022-01-31 12:00:00,2019-10-25 00:00:00,2022-03-28 00:00:00,2021-12-22 12:00:00,0.000000,0.000000e+00,2022-03-30 00:00:00
50%,2022-05-10 00:00:00,2024-03-12 00:00:00,2023-12-11 00:00:00,2022-09-05 00:00:00,2023-11-22 12:00:00,0.000000,0.000000e+00,2023-12-28 12:00:00
75%,2024-04-11 12:00:00,2025-12-25 12:00:00,2024-03-30 00:00:00,2023-07-26 00:00:00,2025-05-05 00:00:00,0.000000,0.000000e+00,2025-05-05 00:00:00
max,2025-12-31 00:00:00,2032-12-31 00:00:00,2024-08-01 00:00:00,2024-12-02 00:00:00,2025-09-29 00:00:00,250.000000,2.817381e+06,2025-09-29 00:00:00
std,NaN,NaN,NaN,NaN,NaN,22.930345,1.635122e+05,NaN


In [11]:
# dados descritivos com textos
new_df.describe(include='object')

,idUnico,nome,cep,endereco,descricaoricao,funcaoSocial,metaGlobal,naturezaOutras,descricaoPlanoNacionalPoliticaVinculado,descricaoPopulacaoBeneficiada,observacoesPertinentes,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
count,606,606,606,606,606,606,606,61,606,606,606,606,606,606,606,606,606,606
unique,606,570,82,221,558,447,430,30,58,33,4,48,73,51,15,48,78,461
top,50379.53-54,CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE,,Não informado,Projeto de desenvolvimento do MDR para integra...,Segurança Pública,Escola de Educação Infantil Tipo B,outros,Não informado,Não informado,Não informado,Não informado ou não existe,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,Não informado ou não existe,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 5, 'descricao': 'Administrativo', 'idE...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
freq,1,7,398,328,7,27,33,15,417,501,530,340,75,271,242,100,75,87


## 4. Armazenamento dos dados

In [12]:
# SALVAR OS DADOS NUM BANCO DE DADOS:

# irei utilizar o sqlite3, pois é um banco de dados relacional leve
# que já vem embutido no python, ideal para protótipos e projetos
# pequenos, além de não precisar de um servidor

PATH_DB = 'data/projetosDF.db'


# Iniciando conexão com o banco de dados
conn = sqlite3.connect(PATH_DB)
cursor = conn.cursor()


# Criação da tabela
cursor.execute("""
CREATE TABLE IF NOT EXISTS projetos (
    idUnico TEXT PRIMARY KEY,
    nome TEXT,
    cep TEXT,
    endereco TEXT,
    descricaoricao TEXT,
    funcaoSocial TEXT,
    metaGlobal TEXT,
    dataInicialPrevista DATE,
    dataFinalPrevista DATE,
    dataInicialEfetiva DATE,
    dataFinalEfetiva DATE,
    dataCadastro DATE,
    especie TEXT,
    natureza TEXT,
    naturezaOutras TEXT,
    situacao TEXT,
    descricaoPlanoNacionalPoliticaVinculado TEXT,
    uf TEXT,
    quantidadeEmpregosGerados REAL,
    descricaoPopulacaoBeneficiada TEXT,
    populacaoBeneficiada REAL,
    observacoesPertinentes TEXT,
    ehModeladaPorBim INTEGER,
    dataSituacao DATE,
    tomadores TEXT,
    executores TEXT,
    repassadores TEXT,
    eixos TEXT,
    tipos TEXT,
    subTipos TEXT,
    fontesDeRecurso TEXT
)
""")

conn.commit()

In [13]:
# MANTER CONSISTENCIA DOS DADOS AO INSERIR NO BANCO:

# Converter NaT em None e datas em texto ISO
for col in new_df.select_dtypes(include=['datetime64[ns]']).columns:
    new_df[col] = new_df[col].apply(lambda x: x.strftime('%Y-%m-%d') if pd.notnull(x) else None)

# Booleanos como inteiros
new_df['ehModeladaPorBim'] = new_df['ehModeladaPorBim'].astype(int)


# Outras colunas: substituir valores nulos por string vazia
new_df = new_df.replace({np.nan: "", "Não informado": "", "nan": ""})


for _, row in new_df.iterrows():
    cursor.execute("""
        INSERT OR REPLACE INTO projetos VALUES (
            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
        )
    """, tuple(row))

conn.commit()
conn.close()



In [14]:
# testando se os dados foram inseridos corretamente
conn = sqlite3.connect(PATH_DB)
cursor = conn.cursor()

cursor.execute('SELECT idUnico, nome FROM projetos LIMIT 5')
for row in cursor.fetchall():
    print(row)

conn.close()

('50379.53-54', 'DL - 304/2024 - Contratação de instituição para execução de serviços técnico-especializados para realização de atualizações no Método de Dimensionamento de Pavimentos Rígidos do DNI')
('42724.53-27', 'Escola Classe Crixá São Sebastião')
('19970.53-78', 'Reajuste do Contrato 45/2021 - Contrução do Centro de Formação e Aperfeiçoamento de Praças - CEFAP do CBMDF')
('24797.53-15', 'Implantação de Passarelas nas Estradas Parque do DF')
('24822.53-70', 'obra de construção da  Cabine de Medição, localizado no Setor Central do Campus Universitário Darcy Ribeiro, da Universidade de Brasília, em Brasília/DF')


In [15]:
conn = sqlite3.connect(PATH_DB)
cursor = conn.cursor()

cursor.execute("SELECT cep, COUNT(*) FROM projetos GROUP BY cep ORDER BY COUNT(*) DESC LIMIT 10")
for row in cursor.fetchall():
    print(row)

conn.close()

('', 398)
('70040902', 25)
('71607900', 14)
('70067901', 14)
('70297400', 11)
('73380900', 9)
('70770917', 7)
('72429005', 5)
('70150900', 5)
('70070906', 5)
